# 🧠 Step 9 — Binary Classification (UP / DOWN)

## Why Binary Classification?

Predicting the exact log return value is extremely difficult — even hedge funds
with billions of dollars of data struggle to do this accurately.

Predicting **direction** (will the market go UP or DOWN tomorrow?) is a much
more tractable problem and directly optimises what matters for trading.

```
Previous approach : predict log_return value → Huber loss → model predicts ~0
This approach     : predict UP(1) or DOWN(0) → Binary Cross Entropy → learns direction
```

## Architecture
```
Numerical features (indicators)     FinBERT embeddings + Sentiment scores
        ↓                                       ↓
  Linear Transformer                    Projection layer
  (temporal patterns)                   (768+5 → 64 dims)
        ↓                                       ↓
        └──────────── CONCATENATE ──────────────┘
                           ↓
             Feedforward Neural Network
                           ↓
                  Sigmoid output (0 to 1)
                  > 0.5 = UP, ≤ 0.5 = DOWN
```

## Files needed (5)
1. `nifty50_step4_standard_train.csv`
2. `nifty50_step4_standard_test.csv`
3. `step7b_finbert_embeddings.csv`
4. `step7a_finbert_sentiment_comparison.csv`
5. `standard_scaler.pkl`

---
**Enable GPU:** Runtime → Change runtime type → T4 GPU → Save

Then: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install torch pandas numpy scikit-learn joblib -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports and Configuration
# ─────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib
import os
import math
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if str(device) == 'cpu':
    print('  ⚠️  No GPU — enable via Runtime → Change runtime type → T4 GPU')

# ── MODEL CONFIG ──────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 5       # short window — captures recent momentum
EMBED_DIM       = 64      # transformer embedding dimension
NUM_HEADS       = 4       # attention heads
NUM_LAYERS      = 2       # transformer layers
TEXT_PROJ_DIM   = 32      # FinBERT 768+5 → 32 projection
HIDDEN_DIM      = 128     # FNN hidden layer
DROPOUT         = 0.2     # balanced dropout
LEARNING_RATE   = 1e-4
EPOCHS          = 150
BATCH_SIZE      = 16
PATIENCE        = 30      # more patience for balanced learning

print(f'\n  Config:')
print(f'    Task            : Binary classification (UP=1 / DOWN=0)')
print(f'    Loss            : Binary Cross Entropy')
print(f'    Sequence length : {SEQUENCE_LENGTH} days')
print(f'    Epochs          : {EPOCHS} (patience={PATIENCE})')
print(f'    Dropout         : {DROPOUT}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload and Load All 5 Input Files
# ─────────────────────────────────────────────────────────────────────────

print('Upload 1/5: nifty50_step4_standard_train.csv')
u = files.upload(); train_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(train_num_df)} train rows')

print('\nUpload 2/5: nifty50_step4_standard_test.csv')
u = files.upload(); test_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(test_num_df)} test rows')

print('\nUpload 3/5: step7b_finbert_embeddings.csv')
u = files.upload(); emb_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(emb_df)} embedding rows')

print('\nUpload 4/5: step7a_finbert_sentiment_comparison.csv')
u = files.upload(); sent_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(sent_df)} sentiment rows')

print('\nUpload 5/5: standard_scaler.pkl')
u = files.upload(); scaler = joblib.load(list(u.keys())[0])
print(f'  ✓ Scaler loaded')

# ── Parse dates ────────────────────────────────────────────────────────────
for df in [train_num_df, test_num_df]:
    df['date'] = pd.to_datetime(
        df['Date'], dayfirst=True, errors='coerce'
    ).dt.strftime('%Y-%m-%d')

emb_df['date'] = pd.to_datetime(
    emb_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

sent_df['date'] = pd.to_datetime(
    sent_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')
sent_df = sent_df.dropna(subset=['date'])
for col in ['finbert_score','gpt4o_score',
            'finbert_pos_prob','finbert_neg_prob','finbert_neu_prob']:
    if col in sent_df.columns:
        sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce').fillna(0)

# ── Feature columns ────────────────────────────────────────────────────────
EXCLUDE_COLS  = ['Date', 'date', 'Close', 'LogReturn_Close']
NUM_FEAT_COLS = [c for c in train_num_df.columns if c not in EXCLUDE_COLS]
TARGET_COL    = 'LogReturn_Close'   # used to derive binary label
EMB_COLS      = [c for c in emb_df.columns if c.startswith('emb_')]
SENT_COLS     = [c for c in ['finbert_score','gpt4o_score',
                              'finbert_pos_prob','finbert_neg_prob',
                              'finbert_neu_prob']
                 if c in sent_df.columns]

print(f'\n  Numerical features : {len(NUM_FEAT_COLS)}')
print(f'  Embedding dims     : {len(EMB_COLS)}')
print(f'  Sentiment features : {SENT_COLS}')
print(f'  Train dates        : {train_num_df["date"].iloc[0]} → {train_num_df["date"].iloc[-1]}')
print(f'  Test dates         : {test_num_df["date"].iloc[0]} → {test_num_df["date"].iloc[-1]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Merge Features and Create Binary Labels
# ─────────────────────────────────────────────────────────────────────────

def build_dataset(num_df, emb_df, sent_df, num_cols, emb_cols, sent_cols, target_col):
    # Merge numerical + embeddings
    cols_needed = ['date', target_col] + [c for c in num_cols if c != target_col]
    merged = pd.merge(
        num_df[cols_needed],
        emb_df[['date'] + emb_cols],
        on='date', how='left'
    )
    # Merge sentiment scores
    if sent_cols:
        merged = pd.merge(
            merged,
            sent_df[['date'] + sent_cols],
            on='date', how='left'
        )
        merged[sent_cols] = merged[sent_cols].fillna(0)

    # Fill missing embeddings with 0
    merged[emb_cols] = merged[emb_cols].fillna(0)

    # Drop rows with NaN in numerical features
    merged = merged.dropna(subset=num_cols).reset_index(drop=True)

    # ── Create binary label ────────────────────────────────────────────────
    # 1 = market UP (log return > 0)
    # 0 = market DOWN (log return <= 0)
    merged['label'] = (merged[target_col] > 0).astype(int)

    return merged


train_df = build_dataset(train_num_df, emb_df, sent_df,
                         NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)
test_df  = build_dataset(test_num_df,  emb_df, sent_df,
                         NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)

# Check class balance
train_up   = train_df['label'].mean() * 100
test_up    = test_df['label'].mean()  * 100

print(f'  Train : {len(train_df)} rows  UP={train_up:.1f}%  DOWN={100-train_up:.1f}%')
print(f'  Test  : {len(test_df)} rows   UP={test_up:.1f}%   DOWN={100-test_up:.1f}%')

# Class balance
n_pos      = train_df['label'].sum()
n_neg      = len(train_df) - n_pos
POS_WEIGHT = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f'  n_pos={n_pos}  n_neg={n_neg}  pos_weight={POS_WEIGHT.item():.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Build Sequence Dataset
# ─────────────────────────────────────────────────────────────────────────

class StockDirectionDataset(Dataset):
    def __init__(self, df, num_cols, emb_cols, sent_cols, seq_len):
        self.seq_len  = seq_len
        self.num_data = df[num_cols].values.astype(np.float32)
        # Combine embeddings + sentiment into one textual vector
        all_text_cols = emb_cols + sent_cols
        self.txt_data = df[all_text_cols].values.astype(np.float32)
        self.labels   = df['label'].values.astype(np.float32)
        self.n        = len(df) - seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        num_seq = self.num_data[idx : idx + self.seq_len]   # (seq_len, num_feats)
        txt_vec = self.txt_data[idx + self.seq_len - 1]     # last day's text vector
        label   = self.labels[idx + self.seq_len]            # next day's direction
        return (
            torch.tensor(num_seq, dtype=torch.float32),
            torch.tensor(txt_vec, dtype=torch.float32),
            torch.tensor(label,   dtype=torch.float32),
        )


ALL_TEXT_COLS = EMB_COLS + SENT_COLS

train_dataset = StockDirectionDataset(
    train_df, NUM_FEAT_COLS, EMB_COLS, SENT_COLS, SEQUENCE_LENGTH
)
test_dataset  = StockDirectionDataset(
    test_df,  NUM_FEAT_COLS, EMB_COLS, SENT_COLS, SEQUENCE_LENGTH
)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

NUM_FEATURES  = len(NUM_FEAT_COLS)
TXT_DIM       = len(ALL_TEXT_COLS)

print(f'  Train sequences  : {len(train_dataset)}')
print(f'  Test sequences   : {len(test_dataset)}')
print(f'  Num features     : {NUM_FEATURES}')
print(f'  Text vector dim  : {TXT_DIM}  (768 emb + {len(SENT_COLS)} sentiment)')
print(f'  Sequence length  : {SEQUENCE_LENGTH} days')

# Verify shapes
nb, tb, lb = next(iter(train_loader))
print(f'\n  Batch shapes:')
print(f'    Numerical : {nb.shape}  (batch, seq_len, num_features)')
print(f'    Text vec  : {tb.shape}  (batch, txt_dim)')
print(f'    Label     : {lb.shape}  (batch,)  values: {lb[:5].tolist()}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Define Binary Classification Model
# ─────────────────────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class StockDirectionClassifier(nn.Module):
    def __init__(self, num_features, txt_dim, embed_dim, num_heads,
                 num_layers, text_proj_dim, hidden_dim, dropout):
        super().__init__()

        # ── Numerical branch: Transformer ─────────────────────────────────
        self.num_proj    = nn.Linear(num_features, embed_dim)
        self.pos_enc     = PositionalEncoding(embed_dim, dropout=dropout)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ── Textual branch: FinBERT + sentiment projection ────────────────
        self.txt_proj    = nn.Sequential(
            nn.Linear(txt_dim, text_proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Fusion FNN ────────────────────────────────────────────────────
        fusion_dim = embed_dim + text_proj_dim
        self.fnn   = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            # No Sigmoid here — BCEWithLogitsLoss applies it internally
            # This is numerically more stable
        )

    def forward(self, num_seq, txt_vec):
        x      = self.num_proj(num_seq)
        x      = self.pos_enc(x)
        x      = self.transformer(x)
        x      = x.mean(dim=1)
        t      = self.txt_proj(txt_vec)
        fused  = torch.cat([x, t], dim=1)
        return  self.fnn(fused).squeeze(-1)


model = StockDirectionClassifier(
    num_features  = NUM_FEATURES,
    txt_dim       = TXT_DIM,
    embed_dim     = EMBED_DIM,
    num_heads     = NUM_HEADS,
    num_layers    = NUM_LAYERS,
    text_proj_dim = TEXT_PROJ_DIM,
    hidden_dim    = HIDDEN_DIM,
    dropout       = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total trainable parameters: {total_params:,}')
print(model)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Train the Model
# ─────────────────────────────────────────────────────────────────────────

# BCEWithLogitsLoss = BCE + Sigmoid fused — more numerically stable
# pos_weight forces model to pay equal attention to UP and DOWN
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Warmup + cosine annealing
WARMUP = 10
def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_loss  = float('inf')
best_val_acc   = 0.0
patience_count = 0
best_epoch     = 0
train_losses   = []
val_losses     = []
val_accs       = []

print('=' * 60)
print('  TRAINING — Binary Classification')
print('=' * 60)
print(f'  Loss      : Binary Cross Entropy')
print(f'  Optimiser : Adam (lr={LEARNING_RATE}, weight_decay=1e-4)')
print(f'  Warmup    : {WARMUP} epochs')
print(f'  Patience  : {PATIENCE} epochs')
print()

for epoch in range(1, EPOCHS + 1):

    # Training
    model.train()
    train_loss = 0.0
    for num_seq, txt_vec, label in train_loader:
        num_seq = num_seq.to(device)
        txt_vec = txt_vec.to(device)
        label   = label.to(device)
        optimizer.zero_grad()
        pred   = model(num_seq, txt_vec)
        loss   = criterion(pred, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss  = 0.0
    val_preds = []
    val_labs  = []
    with torch.no_grad():
        for num_seq, txt_vec, label in test_loader:
            num_seq = num_seq.to(device)
            txt_vec = txt_vec.to(device)
            label   = label.to(device)
            pred    = model(num_seq, txt_vec)
            loss    = criterion(pred, label)
            val_loss += loss.item()
            # Apply sigmoid manually since model no longer has it
            prob = torch.sigmoid(pred).cpu().numpy()
            val_preds.extend((prob > 0.5).astype(int))
            val_labs.extend(label.cpu().numpy().astype(int))
    val_loss /= len(test_loader)
    val_acc   = accuracy_score(val_labs, val_preds) * 100

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step()

    # Print every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:>3}/{EPOCHS}  '
              f'Train={train_loss:.4f}  '
              f'Val={val_loss:.4f}  '
              f'Acc={val_acc:.1f}%  '
              f'LR={lr:.2e}')

    # Early stopping on validation loss
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_val_acc   = val_acc
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), 'step9_model_weights.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  Early stopping at epoch {epoch}')
            print(f'  Best epoch    : {best_epoch}')
            print(f'  Best val loss : {best_val_loss:.4f}')
            print(f'  Best val acc  : {best_val_acc:.1f}%')
            break

print(f'\n✅ Training complete')
print(f'  Best epoch   : {best_epoch}')
print(f'  Best val acc : {best_val_acc:.1f}%')

# Load best weights
model.load_state_dict(torch.load('step9_model_weights.pth', map_location=device))
print('  Best weights loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluate Model
# ─────────────────────────────────────────────────────────────────────────

model.eval()
all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for num_seq, txt_vec, label in test_loader:
        num_seq = num_seq.to(device)
        txt_vec = txt_vec.to(device)
        logits  = model(num_seq, txt_vec)
        prob    = torch.sigmoid(logits).cpu().numpy()
        pred    = (prob > 0.5).astype(int)
        all_probs.extend(prob)
        all_preds.extend(pred)
        all_labels.extend(label.numpy().astype(int))

probs  = np.array(all_probs)
preds  = np.array(all_preds)
labels = np.array(all_labels)

# Metrics
acc      = accuracy_score(labels, preds) * 100
try:
    auc  = roc_auc_score(labels, probs)
except:
    auc  = np.nan
cm       = confusion_matrix(labels, preds)

# Baseline: always predict majority class
majority = int(labels.mean() > 0.5)
baseline = max(labels.mean(), 1 - labels.mean()) * 100

print('=' * 60)
print('  EVALUATION METRICS — Binary Classification')
print('=' * 60)
print(f'  Directional Accuracy : {acc:.2f}%')
print(f'  Naive baseline       : {baseline:.2f}%  (always predict majority)')
print(f'  Improvement over baseline: {acc - baseline:+.2f}%')
print(f'  ROC-AUC              : {auc:.4f}  (0.5=random, 1.0=perfect)')
print(f'  Best epoch           : {best_epoch}')
print()
print('  Confusion Matrix:')
print(f'                  Predicted DOWN  Predicted UP')
print(f'  Actual DOWN  :       {cm[0][0]:>5}           {cm[0][1]:>5}')
print(f'  Actual UP    :       {cm[1][0]:>5}           {cm[1][1]:>5}')
print()
print('  Classification Report:')
print(classification_report(labels, preds,
      target_names=['DOWN (0)', 'UP (1)']))

if acc > baseline:
    print(f'  ✅ Model beats naive baseline by {acc-baseline:+.2f}%')
else:
    print(f'  ⚠️  Model does not beat naive baseline — needs more data or tuning')

# Save metrics
metrics_df = pd.DataFrame([{
    'model':             'Multimodal Binary Classifier (Transformer + FinBERT + FNN)',
    'Directional_Acc':   round(acc, 2),
    'Naive_Baseline':    round(baseline, 2),
    'Improvement':       round(acc - baseline, 2),
    'ROC_AUC':           round(auc, 4) if not np.isnan(auc) else None,
    'Best_Epoch':        best_epoch,
    'Best_Val_Loss':     round(best_val_loss, 4),
}])
metrics_df.to_csv('step9_evaluation_metrics.csv', index=False)
print(f'\n  Metrics saved → step9_evaluation_metrics.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Save Predictions and Training Curve
# ─────────────────────────────────────────────────────────────────────────

test_dates = test_df['date'].values[SEQUENCE_LENGTH:]
test_dates = test_dates[:len(preds)]

pred_df = pd.DataFrame({
    'date':             [pd.to_datetime(d).strftime('%d-%b-%Y') for d in test_dates],
    'actual_label':     labels,
    'actual_direction': ['UP' if l == 1 else 'DOWN' for l in labels],
    'pred_prob':        probs.round(4),
    'pred_label':       preds,
    'pred_direction':   ['UP' if p == 1 else 'DOWN' for p in preds],
    'correct':          (labels == preds),
})

pred_df.to_csv('step9_predictions.csv', index=False)

loss_df = pd.DataFrame({
    'epoch':      list(range(1, len(train_losses) + 1)),
    'train_loss': train_losses,
    'val_loss':   val_losses,
    'val_acc':    val_accs,
})
loss_df.to_csv('step9_training_loss.csv', index=False)

print('  Sample predictions (first 10):')
print(pred_df[['date','actual_direction','pred_direction',
               'pred_prob','correct']].head(10).to_string(index=False))

# High confidence predictions
high_conf = pred_df[abs(pred_df['pred_prob'] - 0.5) > 0.2]
if len(high_conf) > 0:
    hc_acc = high_conf['correct'].mean() * 100
    print(f'\n  High confidence predictions (prob > 0.7 or < 0.3):')
    print(f'    Count    : {len(high_conf)}')
    print(f'    Accuracy : {hc_acc:.1f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 10 — Download All Output Files
# ─────────────────────────────────────────────────────────────────────────
output_files = [
    ('step9_predictions.csv',        'Predicted vs actual direction per date'),
    ('step9_evaluation_metrics.csv', 'Accuracy, AUC, confusion matrix'),
    ('step9_training_loss.csv',      'Train/val loss and accuracy per epoch'),
    ('step9_model_weights.pth',      'Saved best model weights'),
]

print('=' * 60)
print('  STEP 9 COMPLETE — BINARY CLASSIFICATION')
print('=' * 60)
for fname, desc in output_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f'  ✓ {fname}')
        print(f'      {desc}  ({size/1024:.1f} KB)')

print()
print('  Final Results:')
print(f'    Directional Accuracy : {acc:.2f}%')
print(f'    Naive baseline       : {baseline:.2f}%')
print(f'    Improvement          : {acc - baseline:+.2f}%')
print(f'    ROC-AUC              : {auc:.4f}')
print(f'    Best epoch           : {best_epoch}')

print()
print('⬇️  Downloading...')
for fname, _ in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ✓ {fname}')

print()
print('✅ Done! Use these files in your thesis results chapter:')
print('  → step9_evaluation_metrics.csv  — accuracy and AUC table')
print('  → step9_predictions.csv         — prediction vs actual chart')
print('  → step9_training_loss.csv       — training curve plot')